# 03 — Evaluasi terhadap anotasi manusia

Membandingkan kandidat DeepForest (hasil notebook 02, atau dijalankan ulang bila belum ada) dengan anotasi
manusia pada foto contoh yang **punya ground truth**. Pencocokan greedy per confidence, IoU ≥ 0.5.

Batas tafsir: hanya 3 foto hutan NEON yang berasal dari ekosistem/sensor yang sama dengan data pengembang
model → **tidak independen**; bukan akurasi untuk lahan pertanian atau foto K919. Foto tanpa anotasi tidak
dihitung metriknya.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## 1. Prediksi + ground truth

In [ ]:
from wallacea_ml import backend, pretrained
from wallacea_ml.samples import load_samples
from wallacea_ml.voc import load_voc
samples = {s.id: s for s in load_samples()}
if pretrained.DEFAULT_OUT.is_file():
    result = pretrained.load()
    print("memakai hasil notebook 02:", result["created_at"], result["model_version"])
else:
    det = backend.build_api_deepforest(backend.default_deepforest_dir(), confidence=0.35, device="cpu")
    pretrained.save(pretrained.run_on_samples(det, list(samples.values())))
    result = pretrained.load()
    print("notebook 02 belum dijalankan — inferensi dijalankan di sini")
VOC_LABELS = {"Tree": "plant", "Alive": "plant", "Dead": "plant"}   # eksplisit; lihat notebook 01
CONF = result["parameters"]["conf"]
data = {**{k: result[k] for k in ("model_id", "model_version", "split", "domain", "iou_threshold")}, "images": []}
for rec in result["images"]:
    s = samples[rec["image"]]
    if s.annotations is None:
        print("tanpa ground truth (dilewati dari metrik):", s.id)
        continue
    voc = load_voc(s.annotations, VOC_LABELS)
    data["images"].append({"image": s.id, "gt": [{"label": l, "bbox": b} for l, b in voc.boxes], "pred": rec["pred"]})
print(len(data["images"]), "foto dengan ground truth; threshold confidence =", CONF)

## 2. Metrik per foto & gabungan (IoU ≥ 0.5)

In [ ]:
from wallacea_ml.evaluate import evaluate, sweep
r = evaluate(data, leaks=[], conf_threshold=CONF)
display(_df([{k: row[k] for k in ("image", "gt", "pred", "tp", "fp", "fn", "count_error")} for row in r.per_image]))
print({k: (round(getattr(r, k), 3) if isinstance(getattr(r, k), float) else getattr(r, k))
       for k in ("gt_total", "tp", "fp", "fn", "precision", "recall", "f1", "ap", "count_mae_per_image")})
for c in r.caveats + ["Hanya 3 foto NEON, tidak independen dari data pengembang model. Bukan akurasi K919/pertanian."]:
    print("•", c)

## 3. Pengaruh threshold confidence (ilustrasi pada 3 foto — jangan dipakai memilih threshold produk)

In [ ]:
rows = sweep(data, [round(0.05 * i, 2) for i in range(7, 19)])
c = [x["conf"] for x in rows]
plt.figure(figsize=(7, 3.5))
for k in ("precision", "recall", "f1"):
    plt.plot(c, [x[k] for x in rows], label=k)
plt.axvline(CONF, ls="--", c="k"); plt.xlabel("threshold confidence"); plt.legend(); plt.show()
display(_df(rows))

## 4. False positive (merah) & objek terlewat (kuning); cocok = hijau

In [ ]:
from wallacea_ml.evaluate import match_details
from wallacea_ml.viz import draw_boxes
for im in data["images"]:
    m = match_details(im["gt"], [p for p in im["pred"] if p["confidence"] >= CONF], 0.5, "plant")
    boxes = [("plant", d["bbox"]) for d, _ in m["tp"]] + [("gap", d["bbox"]) for d in m["fp"]] + [("check", g["bbox"]) for g in m["fn"]]
    fp_conf = sorted(round(d["confidence"], 2) for d in m["fp"])
    print(f"{im['image']}: {len(m['tp'])} cocok, {len(m['fp'])} false positive, {len(m['fn'])} terlewat; "
          f"confidence FP median {fp_conf[len(fp_conf) // 2] if fp_conf else '-'}")
    plt.figure(figsize=(9, 9))
    plt.imshow(draw_boxes(backend.load_oriented(samples[im["image"]].path), boxes, max_side=1600))
    plt.title(f"{im['image']} — hijau=cocok, merah=FP, kuning=terlewat"); plt.axis("off"); plt.show()